In [ ]:
pip install ogx_client requests rich

In [ ]:
from ogx_client import OgxClient

client = OgxClient(base_url="http://ogx-custom-server-service.ogx.svc.cluster.local:8321")

In [ ]:
models=client.models.list()

for m in models:
    print(m)

In [ ]:
model_id = "vllm-inference/qwen38-27b"
embedding_model_id = "vllm-embedding/redhataimultilingual-e5-large"
#embedding_model_id = "vllm-embedding/baai-bge-m3"
embedding_model = next(m for m in models if m.id == embedding_model_id)
embedding_dimension = int(embedding_model.custom_metadata["embedding_dimension"])

In [ ]:
client.vector_stores.list()

vector_store = client.vector_stores.create(
    name="my_pgvector_store",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "pgvector",
    },
)
vector_store_id = vector_store.id

In [ ]:
client.vector_stores.list()

In [ ]:
import io, time, requests

if embedding_model_id == "vllm-embedding/baai-bge-m3":
    chunk_size = 1500
    chunk_overlap = 300
elif embedding_model_id == "vllm-embedding/redhataimultilingual-e5-large":
    chunk_size = 500
    chunk_overlap = 100

print("chunk_size = " + str(chunk_size))
print("chunk_overlap = " + str(chunk_overlap))

titles = ["Zonnestelsel", "Zon", "Mercurius (planeet)", "Venus (planeet)", "Aarde (planeet)", "Maan",
          "Mars (planeet)", "Jupiter (planeet)", "Saturnus (planeet)", "Uranus (planeet)",
          "Neptunus (planeet)", "Pluto (dwergplaneet)"]

for title in titles:
    for attempt in range(5):
        r = requests.get("https://nl.wikipedia.org/w/api.php",
                         params={"action": "query", "prop": "extracts", "explaintext": 1,
                                 "redirects": 1, "format": "json", "titles": title},
                         headers={"User-Agent": "ogx-rag-demo/0.1"}, timeout=30)
        if r.status_code != 429:
            break
        time.sleep(10)  # rate-limited by Wikipedia, wait and retry
    page = next(iter(r.json()["query"]["pages"].values()))
    text = page.get("extract", "")
    if not text:
        print(f"skipped {title}: not found")
        continue

    file_info = client.files.create(
        file=(f"{page['title']}.txt", io.BytesIO(text.encode("utf-8"))),
        purpose="assistants",
    )
    vs_file = client.vector_stores.files.create(
        vector_store_id=vector_store_id,
        file_id=file_info.id,
        chunking_strategy={"type": "static",
                           "static": {"max_chunk_size_tokens": chunk_size, "chunk_overlap_tokens": chunk_overlap}},
    )
    print(f"{page['title']}: {vs_file.status} {vs_file.last_error or ''}")
    time.sleep(2)

In [ ]:
# query without a vector store
system_instructions = "You are a precise and reliable AI assistant. Use retrieved context when it is available. If nothing relevant is found, say so clearly."

query = "Wie ontdekte de planeet Neptunus?"


In [ ]:
# without lookup
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
)

from IPython.display import Markdown, display

display(Markdown(response.output_text))

In [ ]:
# with lookup
response = client.responses.create(
    model=model_id,
    input=query,
    instructions=system_instructions,
    tools=[{"type": "file_search", "vector_store_ids": [vector_store_id], "max_num_results": 5}],
    include=["file_search_call.results"],# also return the retrieved chunks
    tool_choice="required", # always use tool
)

for item in response.output:
    if item.type == "file_search_call":
        print(f"Search ({item.status}): {item.queries}")
        for r in item.results or []:
            #print(f"  {r.score:.3f}  {r.filename}: {r.text[:100]!r}")
            print(f"  {r.score:.3f}  {r.filename}: {r.text}")
    elif item.type == "message":
        for part in item.content:
            for a in getattr(part, "annotations", None) or []:
                if a.type == "file_citation":
                    print(f"Cited: {a.filename}")

from IPython.display import Markdown, display

display(Markdown(response.output_text))

In [ ]:
from rich import print as rprint
rprint(response)